# System Administration with the Standard Library

These exercises practice the standard library modules seen in the course: `pathlib`, `shutil` and `tempfile` for files, `os.environ` for the environment, `sys` and `fileinput` for the standard streams, `subprocess` for external commands.

So as not to change anything on the machine, the exercises work on a miniature copy of a server, the `server/` folder, which the next cell creates in the current directory: run it before you start (running it again resets the folder). The notebook runs in Colab as in the Codespace, both of them Ubuntu Linux systems where you are `root`. The cells starting with `!` run shell commands, and the "Running commands" exercise runs Linux commands (`dpkg`, `df`, `ls`): on your own machine, you need a Debian or Ubuntu Linux.

The exercises go from the simplest to the hardest: the last ones are optional if you run out of time.

In [ ]:
import gzip
import os
import shutil
from datetime import UTC, datetime
from pathlib import Path

SERVER = Path("server")

FILES = {
  "etc/hostname": "web01\n",
  "etc/hosts": "127.0.0.1 localhost\n10.0.0.12 web01\n",
  "etc/logrotate.conf": "weekly\nrotate 4\ncompress\n",
  "etc/nginx/nginx.conf": "user www-data;\ninclude /etc/nginx/conf.d/*.conf;\n",
  "etc/nginx/mime.types": "types {\n  text/html html;\n  text/css css;\n}\n",
  "etc/nginx/conf.d/shop.conf": "server {\n  server_name shop.example.com;\n}\n",
  "etc/nginx/conf.d/blog.conf": "server {\n  server_name blog.example.com;\n}\n",
  "etc/ssh/sshd_config": "Include /etc/ssh/sshd_config.d/*.conf\nPermitRootLogin no\n",
  "etc/ssh/sshd_config.d/hardening.conf": "PasswordAuthentication no\nMaxAuthTries 3\n",
  "etc/systemd/journald.conf": "[Journal]\nStorage=persistent\n",
  "etc/cron.d/backup": "0 3 * * * root /opt/backup/run.sh\n",
  "home/README.txt": "One folder per user.\n",
  "home/alice/.bashrc": "export EDITOR=vim\n",
  "home/alice/notes.txt": "Renew the shop's certificate.\n",
  "home/bob/report.md": "# Incident report\n",
  "home/carole/.bashrc": "alias ll='ls -l'\n",
  "home/carole/.ssh/authorized_keys": "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5 carole@laptop\n",
  "var/log/syslog": """\
Oct 12 06:25:01 web01 CRON[2210]: (root) CMD (/opt/backup/run.sh)
Oct 12 06:25:03 web01 systemd[1]: Started nginx.service - A high performance web server.
Oct 12 07:02:11 web01 nginx[2301]: error: upstream timed out while reading response header
Oct 12 07:02:12 web01 kernel: [ 4012.551] eth0: link up
Oct 12 08:15:40 web01 nginx[2301]: error: connect() failed (111: Connection refused)
Oct 12 09:00:01 web01 systemd[1]: logrotate.service: Deactivated successfully.
Oct 12 10:31:57 web01 sshd[3120]: error: maximum authentication attempts exceeded for root
""",
  "var/log/auth.log": """\
Oct 12 10:30:02 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:30:05 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:31:57 web01 sshd[3120]: Failed password for invalid user admin from 203.0.113.7
Oct 12 11:12:40 web01 sshd[3302]: Accepted publickey for carole from 192.0.2.15 port 40112
Oct 12 11:12:40 web01 sshd[3302]: pam_unix(sshd:session): session opened for user carole
""",
}

# The logs of dpkg, the package manager, with their last modification date
DPKG_LOGS = {
  "dpkg.log": (
    datetime(2026, 10, 12, 6, 25, 3, tzinfo=UTC),
    "2026-10-12 06:25:03 status installed nginx:amd64 1.24.0-2ubuntu7\n",
  ),
  "dpkg.log.1": (
    datetime(2026, 10, 5, 6, 25, 41, tzinfo=UTC),
    "2026-10-05 06:25:41 status installed openssl:amd64 3.0.13-0ubuntu3\n",
  ),
  "dpkg.log.2.gz": (
    datetime(2026, 9, 28, 6, 26, 10, tzinfo=UTC),
    "2026-09-28 06:26:10 status installed curl:amd64 8.5.0-2ubuntu10\n",
  ),
  "dpkg.log.3.gz": (
    datetime(2026, 9, 21, 6, 25, 57, tzinfo=UTC),
    "2026-09-21 06:25:57 status installed git:amd64 1:2.43.0-1ubuntu7\n",
  ),
}

shutil.rmtree(SERVER, ignore_errors=True)
for name, content in FILES.items():
  path = SERVER / name
  path.parent.mkdir(parents=True, exist_ok=True)
  path.write_text(content, encoding="utf-8")
(SERVER / "etc" / "nginx" / "sites-enabled").mkdir()
for name, (date, content) in DPKG_LOGS.items():
  path = SERVER / "var" / "log" / name
  data = content.encode()
  if name.endswith(".gz"):
    data = gzip.compress(data, mtime=0)
  path.write_bytes(data)
  os.utime(path, (date.timestamp(), date.timestamp()))

for path in sorted(SERVER.rglob("*")):
  print(path)

## User accounts

The `server/home` folder holds one folder per user, and a `README.txt` file. Write a function `users(home)` that takes this folder as a `Path` and returns the sorted list of user names, that is, of the names of its subfolders. Then show, for each user, whether their folder holds a `.bashrc` file.

In [ ]:
# Your code here

### Additional hints

`iterdir()` gives the folder's items, files included: `is_dir()` keeps the folders only. `.name` gives the last part of a path, and `sorted` sorts. The path of a file of the user is built with the `/` operator, and `exists()` tells whether it exists. You should find `['alice', 'bob', 'carole']`, and a `.bashrc` for alice and carole only.

```python
from pathlib import Path


def users(home: Path) -> list[str]:
  return sorted(... for item in home.iterdir() if ...)


home = Path("server/home")
for name in users(home):
  bashrc = ...
  print(f"{name}: .bashrc {'present' if ... else 'missing'}")
```

### Solution

In [ ]:
from pathlib import Path


def users(home: Path) -> list[str]:
  return sorted(item.name for item in home.iterdir() if item.is_dir())


home = Path("server/home")
print(users(home))
for name in users(home):
  bashrc = home / name / ".bashrc"
  print(f"{name}: .bashrc {'present' if bashrc.exists() else 'missing'}")

## Wildcards and file sizes

In the `server/etc` folder and all its subfolders, at any depth:

1. count the files (not the folders);
2. the next cell defines `conf_size_old(folder)`, which computes the total size, in bytes, of the `.conf` files of a folder and its subfolders; it is written as before `pathlib`, with `os.walk` and `os.path`. Run it, then rewrite it with `pathlib`, as `conf_size(folder)` (the "`os.path` Module" table of the section gives the correspondences);
3. check that both functions return the same size for `server/etc`.

**Useful documentation**

- [`os.walk`](https://docs.python.org/3/library/os.html#os.walk)
- [`os.stat_result`](https://docs.python.org/3/library/os.html#os.stat_result)

In [ ]:
import os


def conf_size_old(folder: str) -> int:
  total = 0
  for root, _subfolders, files in os.walk(folder):
    for name in files:
      if os.path.splitext(name)[1] == ".conf":
        total += os.path.getsize(os.path.join(root, name))
  return total

In [ ]:
# Your code here

### Additional hints

`rglob("*")` walks the whole tree, folders included: `is_file()` keeps the files only. In `conf_size_old`, `os.walk` and the loop over `files` amount to `rglob`, and the test on `splitext` to the `"*.conf"` pattern (or to `path.suffix == ".conf"`); `os.path.getsize(os.path.join(root, name))` becomes `path.stat().st_size`, and the loop that adds up `total`, a `sum`. You should find 11 files, and 230 bytes with both functions.

```python
from pathlib import Path

etc = Path("server/etc")
files = [path for path in etc.rglob("*") if ...]
print(f"{len(files)} files")


def conf_size(folder: Path) -> int:
  return sum(... for conf in folder.rglob(...))


print(conf_size(etc), conf_size_old("server/etc"))
```

### Solution

In [ ]:
from pathlib import Path

etc = Path("server/etc")
files = [path for path in etc.rglob("*") if path.is_file()]
print(f"{len(files)} files")


def conf_size(folder: Path) -> int:
  return sum(conf.stat().st_size for conf in folder.rglob("*.conf"))


print(conf_size(etc), conf_size_old("server/etc"))

## Environment variables

1. The `PATH` environment variable lists the folders where the shell looks for commands, separated by `os.pathsep` (`:` on Linux and macOS, `;` on Windows). Show them, one per line and numbered from 1, flagging those that don't exist on the machine.
2. An environment variable is a string, and `bool("0")` is `True`: a yes/no setting must be interpreted. Write a function `boolean_variable(name, default=False)` that returns `True` if the `name` variable is `"1"`, `"true"` or `"yes"`, `False` if it is `"0"`, `"false"` or `"no"`, ignoring case, `default` if it doesn't exist, and raises a `ValueError` for any other value. Try it on the `APP_DEBUG` variable when missing, then set to `"No"`, then to `"maybe"` (catch the `ValueError` to show its message).
3. Give the `LOG_LEVEL` variable the value `"debug"` in `os.environ`, then check that a command run afterwards sees it, with the `!echo $LOG_LEVEL` cell: a command inherits the environment of the program that runs it.

**Useful documentation**

- [`os.pathsep`](https://docs.python.org/3/library/os.html#os.pathsep)

In [ ]:
# Your code here

### Additional hints

`os.environ` reads like a dictionary: `os.environ["PATH"]` raises a `KeyError` if the variable doesn't exist, `os.environ.get(name)` then returns `None`, and `os.environ[name] = value` changes it for the program and the commands it runs afterwards (`os.environ.pop(name, None)` removes it). `enumerate(..., start=1)` numbers from 1. `value.lower()` puts a string in lowercase, and `value in {"1", "true", "yes"}` tests whether it is in the set. You should see `False` (missing variable), `False` (`"No"`), then the `ValueError`'s message. In a cell, `!` runs a shell command, where `$LOG_LEVEL` is the variable's value: the cell should show `debug`.

```python
import os
from pathlib import Path

for number, folder in enumerate(os.environ["PATH"].split(...), start=1):
  missing = "" if Path(folder).exists() else " (missing)"
  print(f"{number}. {folder}{missing}")


def boolean_variable(name: str, default: bool = False) -> bool:
  value = os.environ.get(name)
  if value is None:
    return ...
  if value.lower() in {"1", "true", "yes"}:
    return True
  if ...:
    return False
  raise ValueError(...)


os.environ.pop("APP_DEBUG", None)
print(boolean_variable("APP_DEBUG"))
os.environ["APP_DEBUG"] = "No"
print(...)
os.environ["APP_DEBUG"] = "maybe"
try:
  print(boolean_variable("APP_DEBUG"))
except ValueError as error:
  print(error)

os.environ[...] = ...
```

### Solution

In [ ]:
import os
from pathlib import Path

for number, folder in enumerate(os.environ["PATH"].split(os.pathsep), start=1):
  missing = "" if Path(folder).exists() else " (missing)"
  print(f"{number}. {folder}{missing}")


def boolean_variable(name: str, default: bool = False) -> bool:
  value = os.environ.get(name)
  if value is None:
    return default
  if value.lower() in {"1", "true", "yes"}:
    return True
  if value.lower() in {"0", "false", "no"}:
    return False
  raise ValueError(f"{name}: yes/no value expected, not {value!r}")


os.environ.pop("APP_DEBUG", None)
print(boolean_variable("APP_DEBUG"))
os.environ["APP_DEBUG"] = "No"
print(boolean_variable("APP_DEBUG"))
os.environ["APP_DEBUG"] = "maybe"
try:
  print(boolean_variable("APP_DEBUG"))
except ValueError as error:
  print(error)

os.environ["LOG_LEVEL"] = "debug"

In [ ]:
!echo $LOG_LEVEL

## Archiving files

Create a `confs.tar.gz` archive that holds all the `.conf` files of `server/etc`, in the same subfolders as in `server/etc`. Proceed step by step:

1. create a temporary folder;
2. copy each `.conf` file into it, at its path relative to `server/etc`, first creating the folders that hold it;
3. create the archive from the temporary folder, before the end of the `with` block, which deletes it.

Finally, show the names of the files the archive holds.

**Useful documentation**

- [`Path.relative_to`](https://docs.python.org/3/library/pathlib.html#pathlib.PurePath.relative_to)
- [`tarfile.open`](https://docs.python.org/3/library/tarfile.html#tarfile.open)
- [`TarFile.getmembers`](https://docs.python.org/3/library/tarfile.html#tarfile.TarFile.getmembers)

In [ ]:
# Your code here

### Additional hints

`path.relative_to(etc)` removes the leading `server/etc` from a path. `mkdir(parents=True, exist_ok=True)` creates a folder and its parents, with no error if it already exists. `shutil.copy2(source, destination)` copies a file with its metadata. `shutil.make_archive("confs", "gztar", root_dir=folder)` creates `confs.tar.gz` with the content of `folder` and returns its name. `tarfile.open(name)` opens the archive (in a `with`), whose `getmembers()` gives the items, and `member.isfile()` tells whether one is a file. You should find the 6 `.conf` files of `server/etc`, prefixed with `./`: `./logrotate.conf`, `./nginx/nginx.conf`…

```python
import shutil
import tarfile
import tempfile
from pathlib import Path

etc = Path("server/etc")
with tempfile.TemporaryDirectory() as temp_dir:
  for conf in etc.rglob("*.conf"):
    copy = Path(temp_dir) / ...
    copy.parent.mkdir(...)
    shutil.copy2(conf, copy)
  archive = shutil.make_archive(...)

with tarfile.open(archive) as tar:
  for member in tar.getmembers():
    if ...:
      print(member.name)
```

### Solution

In [ ]:
import shutil
import tarfile
import tempfile
from pathlib import Path

etc = Path("server/etc")
with tempfile.TemporaryDirectory() as temp_dir:
  for conf in etc.rglob("*.conf"):
    copy = Path(temp_dir) / conf.relative_to(etc)
    copy.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(conf, copy)
  archive = shutil.make_archive("confs", "gztar", root_dir=temp_dir)

with tarfile.open(archive) as tar:
  for member in tar.getmembers():
    if member.isfile():
      print(member.name)

In [ ]:
# A variant with shutil.copytree, which copies the whole tree, ignoring the files that
# don't end with .conf: the archive also keeps the folders without a .conf (cron.d,
# sites-enabled), empty.
import shutil
import tempfile
from pathlib import Path


def ignore_except_confs(folder: str, names: list[str]) -> list[str]:
  return [
    name
    for name in names
    if (Path(folder) / name).is_file() and not name.endswith(".conf")
  ]


with tempfile.TemporaryDirectory() as temp_dir:
  copy = Path(temp_dir) / "etc"
  shutil.copytree("server/etc", copy, ignore=ignore_except_confs)
  print(shutil.make_archive("confs-copytree", "gztar", root_dir=copy))

## Dating logs

The `server/var/log` folder holds the logs of `dpkg`, the package manager: `dpkg.log`, the most recent, and its successive rotations, `dpkg.log.1`, then `dpkg.log.2.gz` and `dpkg.log.3.gz`, already compressed.

Copy them into a new `dated-logs` folder, renaming them `YYYY-MM-DD_hh-mm-ss_dpkg.log.gz`, where the date is their last modification date, in Coordinated Universal Time (UTC). Compress along the way with `gzip` the files that aren't yet (those whose name doesn't end with `.gz`). Finally, show the content of the `dated-logs` folder.

**Useful documentation**

- [`datetime.fromtimestamp`](https://docs.python.org/3/library/datetime.html#datetime.datetime.fromtimestamp)
- [`datetime.strftime`](https://docs.python.org/3/library/datetime.html#datetime.datetime.strftime)
- [`strftime` format codes](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes)
- [`gzip.open`](https://docs.python.org/3/library/gzip.html#gzip.open)
- [`shutil.copyfileobj`](https://docs.python.org/3/library/shutil.html#shutil.copyfileobj)

In [ ]:
# Your code here

### Additional hints

`server/var/log` holds other logs: `glob("dpkg.log*")` gives those of `dpkg` only. `path.stat().st_mtime` is the last modification date, in seconds since January 1st, 1970, which `datetime.fromtimestamp(seconds, UTC)` converts into a UTC date. Its format is `"%Y-%m-%d_%H-%M-%S"`: beware, `%m` is the month and `%M` the minutes. To compress, open the source for binary reading (`"rb"`), the destination with `gzip.open(destination, "wb")`, and `shutil.copyfileobj(source, destination)` copies one into the other. The first file, the oldest, is `2026-09-21_06-25-57_dpkg.log.gz`, and the last one `2026-10-12_06-25-03_dpkg.log.gz`.

```python
import gzip
import shutil
from datetime import UTC, datetime
from pathlib import Path

dated = Path("dated-logs")
dated.mkdir(exist_ok=True)
for log in Path("server/var/log").glob("dpkg.log*"):
  date = datetime.fromtimestamp(..., UTC)
  destination = dated / f"{date.strftime(...)}_dpkg.log.gz"
  if log.suffix == ".gz":
    shutil.copy2(log, destination)
  else:
    with log.open("rb") as source, gzip.open(...) as compressed:
      ...

for path in sorted(dated.iterdir()):
  print(path.name)
```

### Solution

In [ ]:
import gzip
import shutil
from datetime import UTC, datetime
from pathlib import Path

dated = Path("dated-logs")
dated.mkdir(exist_ok=True)
for log in Path("server/var/log").glob("dpkg.log*"):
  date = datetime.fromtimestamp(log.stat().st_mtime, UTC)
  destination = dated / f"{date.strftime('%Y-%m-%d_%H-%M-%S')}_dpkg.log.gz"
  if log.suffix == ".gz":
    shutil.copy2(log, destination)
  else:
    with log.open("rb") as source, gzip.open(destination, "wb") as compressed:
      shutil.copyfileobj(source, compressed)

for path in sorted(dated.iterdir()):
  print(path.name)

## Protecting a secret

Write into `server/home/alice/secret.txt` a random string of 20 letters and digits, drawn with the `secrets` module, made for that (the draws of the `random` module are predictable: it isn't meant for a secret). The file must be created with only the write permission for its owner (`0o200`), then, once the secret is written, have only the read permission for its owner (`0o400`). Show its permissions after the write, then at the end.

A read-only file can't be opened again for writing by its owner: first delete the file if it exists, so that the cell can be run again. Note that Colab and the Codespace have you work as `root`, whom permissions don't stop.

**Useful documentation**

- [`secrets.choice`](https://docs.python.org/3/library/secrets.html#secrets.choice)
- [`string.ascii_letters`](https://docs.python.org/3/library/string.html#string.ascii_letters)
- [`open`'s `opener` parameter](https://docs.python.org/3/library/functions.html#open)
- [`os.open`](https://docs.python.org/3/library/os.html#os.open)
- [`Path.unlink`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.unlink)

In [ ]:
# Your code here

### Additional hints

`secrets.choice(alphabet)` draws a character from `alphabet`, here `string.ascii_letters + string.digits`. `open` creates the file with the default permissions: to choose your own, pass it an `opener`, a function that takes the path and the opening flags and calls `os.open(path, flags, permissions)`. `unlink(missing_ok=True)` deletes a file, with no error if it doesn't exist. `stat.filemode(path.stat().st_mode)` shows the permissions as `ls -l` does: you should see `--w-------`, then `-r--------`. The permissions requested from `os.open` go through the user's mask, `os.umask` (often `0o022`), which removes rights from them: `0o200 & ~0o022` stays `0o200`. `os.umask(mask)` changes the mask and returns the old one.

```python
import os
import secrets
import stat
import string
from pathlib import Path

alphabet = string.ascii_letters + string.digits
value = "".join(... for _ in range(20))

secret = Path("server/home/alice/secret.txt")
secret.unlink(...)
with open(secret, "w", opener=lambda path, flags: os.open(...)) as file:
  file.write(value)
print(stat.filemode(secret.stat().st_mode))
secret.chmod(...)
print(stat.filemode(secret.stat().st_mode))
```

### Solution

In [ ]:
import os
import secrets
import stat
import string
from pathlib import Path

alphabet = string.ascii_letters + string.digits
value = "".join(secrets.choice(alphabet) for _ in range(20))

mask = os.umask(0)  # changes the mask and returns the old one,
os.umask(mask)  # which we put back at once
print(f"umask: {oct(mask)}")

secret = Path("server/home/alice/secret.txt")
secret.unlink(missing_ok=True)
with open(secret, "w", opener=lambda path, flags: os.open(path, flags, 0o200)) as file:
  file.write(value)
print(stat.filemode(secret.stat().st_mode))
secret.chmod(0o400)
print(stat.filemode(secret.stat().st_mode))

## Running commands

Write a function `run_command(command)` that runs a command, given as a list of strings, with `subprocess.run`, and returns its standard output as text. If the command fails (its return code isn't 0), or if the program doesn't exist, it shows the return code and the error message, and returns `None`. Then use it to:

1. count the packages installed on the machine: the lines of `dpkg -l` that start with `ii`;
2. show the space available on the disk of the current directory: the fourth column of the last line of `df -h .`;
3. list a folder that doesn't exist, with `ls missing-folder`;
4. run a program that doesn't exist, `missing-command`.

In [ ]:
# Your code here

### Additional hints

`subprocess.run(command, capture_output=True, text=True, check=True)` captures the outputs instead of showing them (`text=True` decodes them into strings) and raises a `subprocess.CalledProcessError` if the return code isn't 0; the result has the `stdout` and `returncode` attributes, the exception `returncode` and `stderr`. A program that can't be found raises `FileNotFoundError`. `splitlines()` splits a text into lines, `split()` a line into columns. Colab counts several hundred packages; `ls missing-folder` ends with code 2.

```python
import subprocess


def run_command(command: list[str]) -> str | None:
  try:
    result = subprocess.run(command, ...)
  except subprocess.CalledProcessError as error:
    print(f"{command}: return code {...}, error: {...}")
    return None
  except FileNotFoundError:
    print(f"{command}: program not found")
    return None
  return result.stdout


packages = run_command(["dpkg", "-l"])
if packages is not None:
  installed = [line for line in packages.splitlines() if ...]
  print(f"{len(installed)} packages installed")

disk = run_command(["df", "-h", "."])
if disk is not None:
  print(f"Available space: {...}")

run_command(["ls", "missing-folder"])
run_command(["missing-command"])
```

### Solution

In [ ]:
import subprocess


def run_command(command: list[str]) -> str | None:
  try:
    result = subprocess.run(command, capture_output=True, text=True, check=True)
  except subprocess.CalledProcessError as error:
    print(f"{command}: return code {error.returncode}, error: {error.stderr.strip()}")
    return None
  except FileNotFoundError:
    print(f"{command}: program not found")
    return None
  return result.stdout


packages = run_command(["dpkg", "-l"])
if packages is not None:
  installed = [line for line in packages.splitlines() if line.startswith("ii")]
  print(f"{len(installed)} packages installed")

disk = run_command(["df", "-h", "."])
if disk is not None:
  print(f"Available space: {disk.splitlines()[-1].split()[3]}")

run_command(["ls", "missing-folder"])
run_command(["missing-command"])

## A command-line tool

Write the `search.py` script, a small `grep`: `python search.py PATTERN [FILE...]` prints the lines of the given files that contain the string `PATTERN`, or those of the standard input if no file is given, so that it can be used in a pipe like the Unix commands. Moreover:

- when the `SEARCH_VERBOSE` environment variable is `1`, each line is preceded by the name of its file and its line number in that file, as `file:number:line`;
- at the end, the script logs the number of lines found with `logging.info`, which writes to the standard error, so as not to mix it with the results that a pipe passes on to the next command; the logging level comes from the `LOG_LEVEL` environment variable (`WARNING` by default, which hides the `INFO` messages);
- without a pattern, it logs its usage with `logging.error` and exits with return code 2; otherwise, it exits with code 0 if it found a line, 1 otherwise, as `grep` does.

Give it the shape of the slides' `count.py` tool ("A System Tool With `sys`"): a `main()` function that returns the return code, called by `sys.exit(main())` under `if __name__ == "__main__":`.

The exercise cell writes the script with `%%writefile`; the next three cells try it on a file at the `INFO` level, on two files with the `SEARCH_VERBOSE` variable at the `WARNING` level, then in a pipe at the `INFO` level.

In [ ]:
%%writefile search.py
# Your code here

In [ ]:
!LOG_LEVEL=info python search.py error server/var/log/syslog

In [ ]:
!SEARCH_VERBOSE=1 LOG_LEVEL=warning python search.py Failed server/var/log/auth.log server/var/log/syslog

In [ ]:
!cat server/var/log/syslog server/var/log/auth.log | LOG_LEVEL=info python search.py sshd | wc -l

### Additional hints

`sys.argv` is the list of the command line's arguments, the script's name included: `sys.argv[1]` is the pattern, `sys.argv[2:]` the files. `fileinput.input(files=sys.argv[2:], encoding="utf-8")` gives the lines of these files one after the other, or those of the standard input if the list is empty; during the loop, `fileinput.filename()` and `fileinput.filelineno()` give the file and the number of the current line. `logging.basicConfig(level=os.environ.get("LOG_LEVEL", "WARNING").upper(), format="%(levelname)s %(message)s")` sets the level from which messages are shown, and `logging.info(...)` and `logging.error(...)` write them to the standard error. `main()` returns the return code, and `sys.exit(main())` stops the script with this code. The first try finds the 3 `error` lines of `syslog`, followed by `INFO 3 line(s) found`. The second one finds the 3 `Failed` lines of `auth.log`, from `server/var/log/auth.log:1:` to `server/var/log/auth.log:3:`, with no `INFO` message, hidden at the `WARNING` level. The third one finds 6 lines: `wc -l`, which counts the lines it receives through the pipe, shows 6, and the `INFO 6 line(s) found` message, written to the standard error, doesn't go through the pipe.

```python
%%writefile search.py
import fileinput
import logging
import os
import sys


def main() -> int:
  logging.basicConfig(level=..., format="%(levelname)s %(message)s")
  if len(sys.argv) < 2:
    logging.error(...)
    return ...
  pattern = sys.argv[1]
  verbose = os.environ.get(...) == "1"
  found = 0
  for line in fileinput.input(files=..., encoding="utf-8"):
    if ...:
      found += 1
      if verbose:
        print(f"{fileinput.filename()}:{...}:", end="")
      print(line, end="")
  logging.info(...)
  return ...


if __name__ == "__main__":
  sys.exit(main())
```

### Solution

In [ ]:
%%writefile search.py
import fileinput
import logging
import os
import sys


def main() -> int:
  logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "WARNING").upper(),
    format="%(levelname)s %(message)s",
  )
  if len(sys.argv) < 2:
    logging.error("usage: %s PATTERN [FILE...]", sys.argv[0])
    return 2
  pattern = sys.argv[1]
  verbose = os.environ.get("SEARCH_VERBOSE") == "1"
  found = 0
  for line in fileinput.input(files=sys.argv[2:], encoding="utf-8"):
    if pattern in line:
      found += 1
      if verbose:
        print(f"{fileinput.filename()}:{fileinput.filelineno()}:", end="")
      print(line, end="")
  logging.info("%d line(s) found", found)
  return 0 if found else 1


if __name__ == "__main__":
  sys.exit(main())